# 06 Deterministic local fallback
## Learning objective
Run the real governed package, approval boundary, DRY_RUN writer, audit and evaluations without Azure.
## Technical level
L200-L400. Expected duration: 20 minutes.
## Prerequisites and setup
Python 3.12, uv, `.venv` selected in VS Code. Bash/PowerShell: `uv sync --frozen --all-extras --dev`. Install once before disconnecting. This is not a zero-install/standard-library-only runtime. Read the [notebook guide](README.md) for lab order, consent rules and the production/WAF disclaimer.
## Cost and permissions
No Azure credentials, license, subscription or service costs. Local synthetic policy only. No email-retention or real organizational guidance.


In [ ]:
from pathlib import Path
from uuid import uuid4

from foundry_security_accelerator.audit import verify_audit
from foundry_security_accelerator.config import Settings
from foundry_security_accelerator.contracts import UserRequest
from foundry_security_accelerator.redteam import load_items, reconcile_items
from foundry_security_accelerator.workflow import run
from foundry_security_accelerator.workflow_evaluation import evaluate_workflow

ROOT = Path.cwd() if Path("pyproject.toml").exists() else Path.cwd().parent
assert (ROOT / "pyproject.toml").exists()
directory = ROOT / ".fsa" / "notebooks" / str(uuid4())
settings = Settings.local()
request = UserRequest(text="Explain synthetic equipment policy.", subject="synthetic-analyst")
pending = run(request, directory, settings=settings)
assert pending["status"] == "AWAITING_APPROVAL"
assert pending["telemetry_export"] == "LOCAL_ONLY"
assert verify_audit(Path(pending["audit"]), pending["audit_head"])
approved = run(
    UserRequest(text=request.text, subject=request.subject),
    directory,
    settings=settings,
    demo_approval=True,
)
assert approved["status"] == "DRY_RUN"
assert verify_audit(Path(approved["audit"]), approved["audit_head"])
print(
    {
        "pending": pending["status"],
        "synthetic_approved": approved["status"],
        "export": approved["telemetry_export"],
    }
)

In [ ]:
report = evaluate_workflow(
    ROOT / "data/evaluation/workflow.jsonl", ROOT / "data/evaluation/thresholds.toml", directory
)
assert report["status"] == "PASS"
assert report["denominators"]["task_adherence_rate"] == 16
raw = load_items(ROOT / "data/redteam/synthetic_items.json")
before = raw.model_dump_json()
reconciled = reconcile_items(raw)
assert raw.model_dump_json() == before
assert reconciled["strict_failed"] is True
assert reconciled["summary"]["outcome_counts"]["UNKNOWN"] == 1
print(
    {"local_gate": report["status"], "strict_redteam": "EXPECTED_REJECTION", "raw_preserved": True}
)

## Today's D8b lifecycle walkthrough
This is a local state-machine simulation of request, review, independent approval, DRY_RUN publication and a rejected update. It creates no registry entry, consent, Conditional Access policy, access package or identity. Portal evidence remains separate.


In [ ]:
from foundry_security_accelerator.lifecycle import run_lifecycle

lifecycle = run_lifecycle(directory / "lifecycle-demo")
assert lifecycle["status"] == "PASS"
print({"kind": lifecycle["kind"], "status": lifecycle["status"]})

## Expected results and validation
AWAITING_APPROVAL without a reviewer; DRY_RUN with explicit local synthetic approval; verified audit; 16-case local threshold PASS; strict red-team rejection of unsafe/unknown synthetic evidence. CI executes this notebook with service sockets and credential acquisition denied, without saving output. Run `uv run python scripts/validate_notebooks.py --execute-local` to reproduce.
## Security explanation
Retrieval, inspection, fixed-tool authorization and policy are the existing workflow, not duplicated notebook logic. Synthetic roles/approval do not establish Entra/OBO or enterprise human authorization. Raw evaluator records remain unchanged; missing evidence is never PASS. Selected synthetic attacks do not estimate population attack success. Local spans are not native Agent 365, Defender or Purview signals.
## Troubleshooting
Import failure: select `.venv`. Data file error: open from repo/notebooks. A failing assertion is a blocker; do not remove it or claim success. See [troubleshooting](../docs/participant-lab/troubleshooting.md).
## Cleanup
No cloud resources created. Keep original local evidence private until review, remove only your own named run directory when finished, and clear notebook outputs before sharing. [Cleanup](../docs/participant-lab/cleanup.md).
## Source modules
[workflow](../src/foundry_security_accelerator/workflow.py), [evaluation](../src/foundry_security_accelerator/workflow_evaluation.py), [red-team](../src/foundry_security_accelerator/redteam.py), [audit](../src/foundry_security_accelerator/audit.py).
